# Data Science Capstone Project  
## Analysis of Heterogeneity in Antidepressant Treatment Response

**Format:** Google Colab / Jupyter Notebook  
**Project type:** analysis of clinical trial data with individual patient-level data
**Data:** open longitudinal clinical trial dataset `rbmi::antidepressant_data`  
**Main methods:** preprocessing, EDA, investigator-level adjustment, LMM-style model, Gaussian Mixture Models with random starts, quantile regression, bootstrap validation, sensitivity analysis.

---

## Project overview

| Field | Description |
|---|---|
| Program | Data Science Specialist |
| Task type | Tabular clinical data analysis |
| Domain | Clinical tabular data / depression treatment response |
| Main question | Is there evidence of heterogeneity in treatment response, and is it better described by discrete latent subgroups or by a continuous change of the effect across the response distribution? |
| Scientific motivation | Debate between the FMM interpretation of Stone et al. and the QTE interpretation of Meyerson et al. |
| Primary target | `percentage_response = (baseline_score - endpoint_score) / baseline_score` |
| Sensitivity target | `absolute_response = baseline_score - endpoint_score` |
| Primary baseline threshold | `BASVAL >= 14` |
| Grouping structure | `POOLINV`, pooled investigator |
| Models | Adjustment model, FMM, quantile regression |
| Validation | Bootstrap, BIC/AIC, sensitivity analysis, pinball loss, pseudo R² |
| Limitation | A single open dataset is used; this is not an IPD meta-analysis |

---


---

## Project idea in brief

The average treatment effect can hide heterogeneity in individual response. The project therefore compares two interpretations:

1. **Latent response subgroups**: the distribution of responses can be described by several hidden components via Finite Mixture Models.
2. **Continuous heterogeneity**: the treatment effect changes along the distribution of responses; this is tested with quantile regression.

These two models reflect a real scientific debate in the antidepressant literature. Stone et al. proposed an FMM interpretation of latent response groups, while Meyerson et al. proposed a QTE interpretation of a more distributed treatment effect.

The key outcome is to compare which interpretation better and more stably describes the structure of individual response in the available open dataset.

# 1. Problem statement

## 1.1. Why this problem matters

Meta-analyses of antidepressant clinical trials usually estimate the average treatment effect using Cohen's d or the standardized mean difference. But the average effect may be insufficient to understand how the treatment acts on different patients. For example, the same average effect can arise in two different situations:

- the treatment helps almost everyone a little;
- the treatment helps a small subset of patients a lot and barely helps the rest.

For medical decision-making and for future treatment personalization it is important to understand not only the average effect but also the shape of the distribution of individual response.

## 1.2. The scientific debate behind the project

The project is based on the scientific debate in recent meta-analyses.

**Stone et al., BMJ 2022** analyzed individual patient data from 232 placebo-controlled trials submitted to the FDA and used **finite mixture models**. Their interpretation: the response distribution is better described by three components — Large, Non-specific, and Minimal response. In this logic, the average antidepressant effect can arise because active treatment increases the probability of falling into the large-response group.  
Source: [Stone et al., BMJ 2022](https://www.bmj.com/content/378/bmj-2021-067606)

**Meyerson et al., JAMA Network Open 2023** used the same FDA clinical data but a different statistical framework — **quantile treatment effects**. They restricted the analysis to participants with a baseline HAMD score ≥ 20 and showed that the response distribution in the treatment group was better than placebo at every studied quantile, with the largest separation around the 55th quantile. Their interpretation is closer to a continuous heterogeneity of the effect than to rigid latent classes.  
Source: [Meyerson et al., JAMA Network Open 2023](https://jamanetwork.com/journals/jamanetworkopen/fullarticle/2805805)

Additionally, **Xu et al., Journal of Clinical Epidemiology 2025** attempted to verify the trimodal FMM interpretation on STAR*D and were unable to reproduce the components reported by Stone et al.
Source: [Xu et al., Journal of Clinical Epidemiology 2025](https://www.jclinepi.com/article/S0895-4356(25)00276-8/fulltext)

Thus, the main research idea of the project:

> Test on an open dataset whether response heterogeneity looks more like stable latent subgroups with a trimodal response, or like a continuous change of the effect across the distribution.

## 1.3. Project goal

Build a reproducible pipeline for analyzing heterogeneity of antidepressant treatment response in an open clinical dataset.

## 1.4. Project objectives

1. Load the open longitudinal clinical trial dataset.
2. Reshape the data to a patient-level endpoint format.
3. Compute two response targets:
   - percentage response;
   - absolute response.
4. Run EDA and check the structure of the data.
5. Adjust for covariates such as baseline severity and investigator-site heterogeneity via the `POOLINV` variable.
6. Test the latent-subgroup hypothesis with Gaussian Mixture Models.
7. Test continuous heterogeneity of the treatment effect with quantile regression
8. Run bootstrap validation and a sensitivity analysis.
9. Produce an interpretable summary for the project defense.

## 1.5. What the project does NOT claim

This project does not claim that the statistical components found are automatically biological phenotypes. FMM is used as an exploratory tool.

# 2. Study design and methodology

## 2.1. Study format

The capstone project is an analysis of individual patient data from a single open longitudinal dataset with repeated measurements.

Additional open antidepressant clinical-trial data could not be obtained within the project timeframe, so the work is not positioned as a meta-analysis.



## 2.2. The role of the `POOLINV` variable

The source dataset contains a `POOLINV` variable. In the documentation
(https://openpharma.github.io/rbmi/main/reference/antidepressant_data.html) it is described as pooled investigator.

In this project, `POOLINV` is treated as an anonymized identifier of the investigator-structure level within a single clinical trial. It is a grouping variable contained in the source table.

Accounting for `POOLINV` matters because patients in the same group may be similar to each other in recruitment conditions, assessment procedures, and baseline disease severity. If this structure is ignored, part of the differences between investigator groups may be wrongly interpreted as individual heterogeneity of treatment response.

In the project, `POOLINV` is used for three purposes:

1. diagnostics of the internal structure of the dataset;
2. centering of baseline severity within investigator groups;
3. an adjusted model via a random intercept or a fixed effect `C(POOLINV)`.

## 2.3. Baseline centering

In individual-patient-data studies it is important to separate individual and group-level effects of covariates. In Stone et al., covariates were centered at the patient level around the trial mean to reduce the risk of an ecological fallacy in covariate-outcome interactions. Methodologically this relates to the approach of Burke et al.
Sources: [Stone et al., BMJ 2022](https://www.bmj.com/content/378/bmj-2021-067606), [Burke et al., Statistics in Medicine 2017](https://doi.org/10.1002/sim.7141)

This project uses a single dataset, so the same idea is applied not at the level of different trials but at the level of the "POOLINV" variable. For baseline depression severity the following is computed:

```text
BASVAL_centered = BASVAL - mean(BASVAL within POOLINV)
```

This variant lets us compare a patient not only with the overall sample but with patients inside the same investigator group. It reduces the risk that between-group differences are taken for individual differences in response.

## 2.4. Severity threshold and investigator-group adjustment

The severity threshold (baseline cut-off) and the "POOLINV" adjustment solve different methodological problems.

"BASVAL >= 14" is used as the severity cut-off for including patients in the primary analysis. This threshold is not arbitrary: in the Xu et al. re-analysis of STAR*D, which checked the conclusions of Stone et al., the primary analysis also included patients with HDRS ≥ 14, and the authors note that this threshold is the gold standard.
Source: [Xu et al., Journal of Clinical Epidemiology 2025](https://www.jclinepi.com/article/S0895-4356(25)00276-8/fulltext)

The practical role of the threshold in the project is to reduce the instability of the percentage-response variable "percentage_response" for patients with very low baseline severity. With a low baseline score, even a small absolute change can produce a disproportionately large percentage response.


- the severity threshold increases the stability of the target variable percentage response;
- the POOLINV adjustment accounts for the internal grouping of patients.

## 2.5. Quantile regression as a practical adaptation of the QTE approach

 Meyerson et al. proposed an interpretation of antidepressant treatment response via "quantile treatment effects" — the difference between the response distributions of placebo and active treatment at different quantiles.

In this project, quantile regression is used instead of a full QTE because it allows individual covariates to be included.

Model specification

```text
response ~ treatment + BASVAL_centered + gender_female
```

The model uses all available variables suitable for adjustment:

- treatment — drug vs placebo;
- BASVAL_centered — baseline depression severity, centered within POOLINV;
- gender_female — the available demographic covariate.

The variables HAMDTL17, CHANGE, and PGIIMP are not included because they are tied to the endpoint outcome; using them as predictors could lead to data leakage.

A sensitivity specification is additionally checked:

```text
response ~ treatment + BASVAL_centered + gender_female + C(POOLINV)
```

It shows whether the treatment-effect profile is preserved when the investigator group is modeled explicitly.

## 2.6. Spline and ANCOVA as a sensitivity check.

In Stone et al., "multivariable adaptive regression splines" were used for continuous covariates to capture possible non-linear relationships of baseline severity, age, and other covariates with the outcome. In this project the primary model is kept simpler because the filtered sample is small and the set of available covariates is limited.

To check the robustness of the results, a spline specification for baseline severity is added to the sensitivity analysis:

```text
response ~ treatment + spline(BASVAL_centered) + gender_female
```

An ANCOVA-style endpoint model is also added:

```text
endpoint_score ~ treatment + baseline severity + gender + POOLINV
```

This check matters because baseline severity can mechanically correlate with the change in score from the start of treatment.

# 3. Library installation and configuration

This notebook is designed to run in Google Colab. No GPU/TPU is required: the libraries used (`pandas`, `statsmodels`, `sklearn`) mostly run on CPU.

In [ ]:
!pip install -q pyreadr requests pandas numpy scipy scikit-learn statsmodels matplotlib seaborn

In [ ]:
import io
import re
import tarfile
import time
import warnings
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import pyreadr
import requests

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
from sklearn.mixture import GaussianMixture

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from patsy import bs
from typing import Optional

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# primary analysis settings
PRIMARY_BASELINE_CUTOFF = 14
SENSITIVITY_CUTOFFS = [None, 14, 18, 20, 24]

# quantile regression settings
QUANTILES = np.round(np.arange(0.05, 1.00, 0.05), 2)

# FMM and K settings following Stone et al.
FMM_COMPONENTS = range(1, 6)
FMM_COVARIANCE_TYPES_MAIN = ("full", "tied", "diag", "spherical")
FMM_RANDOM_STARTS_MAIN = 100

# Bootstrap settings
# 150 because of limited Colab capacity
FMM_BOOTSTRAP_ITERATIONS = 150
FMM_COVARIANCE_TYPES_BOOTSTRAP = ("full", "tied")
FMM_RANDOM_STARTS_BOOTSTRAP = 100

# Interpretation rule for mixture components
MIN_COMPONENT_WEIGHT = 0.05

GMM_DISPLAY_COLUMNS = [
    "n_components",
    "covariance_type",
    "bic",
    "aic",
    "converged_runs",
    "best_seed",
]

# storage settings
DATA_DIR = Path("data")
FIG_DIR = Path("reports/figures")
RESULTS_DIR = Path("reports/results")

for directory in [DATA_DIR, FIG_DIR, RESULTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# plotting settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8,5)
plt.rcParams["axes.grid"] = True

pd.set_option("display.max_columns",120)
pd.set_option("display.width",160)

print("Project configuration loaded.")
print(f"Primary baseline cut-off: BASVAL >= {PRIMARY_BASELINE_CUTOFF}")
print(f"Bootstrap iterations: {FMM_BOOTSTRAP_ITERATIONS}")
print(f"GMM random starts: {FMM_RANDOM_STARTS_MAIN}")

# 4. Data loading

## 4.1. Data source

The open dataset `antidepressant_data` from the R package `rbmi` is used.

The dataset contains longitudinal clinical trial data:

- `PATIENT` — patient identifier;
- `THERAPY` — DRUG / PLACEBO;
- `BASVAL` — baseline HAMD17;
- `HAMDTL17` — HAMD17 at the visit;
- `CHANGE` — change in HAMD17;
- `VISIT` — visit number;
- `POOLINV` — investigator / site-level grouping variable;
- `GENDER` — patient gender.

In [ ]:
csv_path = Path("antidepressant_data.csv")

if not csv_path.exists():
# link to the ready-made data file
    url = "https://raw.githubusercontent.com/cran/rbmi/master/data/antidepressant_data.rda"
    rda_path = "temp.rda"
    urllib.request.urlretrieve(url, rda_path)
    pyreadr.read_r(rda_path)["antidepressant_data"].to_csv(csv_path, index=False)

df_raw = pd.read_csv(csv_path)
# check
print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
display(df_raw.head())


## Intermediate conclusion

The data loaded successfully; the number of rows is larger than the number of patients because the dataset contains repeated measurements over 7 weeks.

# 5. Project functions

This section collects the functions for data processing, diagnostics, modeling, validation, and saving results. This approach improves code readability and follows the structuring requirement.

In [ ]:
# ================= helpers =================

def save_fig(fn: str) -> None:
    plt.tight_layout()
    plt.savefig(FIG_DIR / fn, dpi=200, bbox_inches="tight")
    print(f"Saved: {FIG_DIR / fn}")


def finalize_plot(title: str, xlabel: str, ylabel: str | None = None,
                  figsize: tuple = (8, 5), filename: Optional[str] = None) -> None:
    plt.gcf().set_size_inches(figsize)
    plt.title(title); plt.xlabel(xlabel)
    if ylabel: plt.ylabel(ylabel)
    if plt.gca().get_legend_handles_labels()[0]: plt.legend()
    if filename: save_fig(filename)
    plt.show()


def pinball_loss(y_true, y_pred, q: float) -> float:
    r = np.asarray(y_true) - np.asarray(y_pred)
    return float(np.mean(np.maximum(q * r, (q - 1) * r)))


def add_fdr(df: pd.DataFrame, p_col: str = "p_value") -> pd.DataFrame:
    df = df.copy()
    mask = df[p_col].notna()
    df = df.assign(p_value_fdr_bh=np.nan, significant_fdr_0_05=False)
    if mask.any():
        rej, corr, _, _ = multipletests(df.loc[mask, p_col].values, alpha=0.05, method="fdr_bh")
        df.loc[mask, ["p_value_fdr_bh", "significant_fdr_0_05"]] = np.column_stack([corr, rej])
    return df


# ================= DATA =================

def prepare_data(data: pd.DataFrame) -> pd.DataFrame:
    df = data.rename(columns=str.upper)
    num = ["HAMATOTL", "PGIIMP", "RELDAYS", "VISIT", "BASVAL", "HAMDTL17", "CHANGE"]
    df[num] = df[num].apply(pd.to_numeric, errors="coerce")

    return (df.astype({"PATIENT": str, "POOLINV": str})
            .assign(THERAPY=lambda x: x.THERAPY.str.upper(),
                    GENDER=lambda x: x.GENDER.str.upper())
            .sort_values(["PATIENT", "VISIT", "RELDAYS"])
            .reset_index(drop=True))


def build_endpoint(data: pd.DataFrame, endpoint: str = "last_available",
                   baseline_cutoff: float | None = None) -> pd.DataFrame:
    if endpoint == "last_available":
        df = data.sort_values(["PATIENT", "VISIT", "RELDAYS"]).groupby("PATIENT").tail(1)
    elif endpoint == "visit7":
        df = data[data["VISIT"] == 7].copy()
    else:
        raise ValueError("endpoint must be 'last_available' or 'visit7'")

    df = df.dropna(subset=["PATIENT", "THERAPY", "GENDER", "POOLINV", "BASVAL", "CHANGE"]).copy()
    if baseline_cutoff is not None:
        df = df[df["BASVAL"] >= baseline_cutoff].copy()

    return df.assign(
        treatment=(df["THERAPY"] == "DRUG").astype(int),
        gender_female=(df["GENDER"] == "F").astype(int),
        absolute_response=-df["CHANGE"],
        percentage_response=-df["CHANGE"] / df["BASVAL"],
        BASVAL_centered=lambda x: x["BASVAL"] - x.groupby("POOLINV")["BASVAL"].transform("mean")
    ).reset_index(drop=True)


def poolinv_diag(data: pd.DataFrame) -> pd.DataFrame:
    agg = data.groupby("POOLINV").agg(
        n=("PATIENT", "count"), n_drug=("treatment", "sum"), mean_base=("BASVAL", "mean")
    ).reset_index()
    return (agg.assign(
                n_placebo=agg["n"] - agg["n_drug"],
                drug_share=agg["n_drug"] / agg["n"],
                has_both=(agg["n_drug"] > 0) & (agg["n"] - agg["n_drug"] > 0)
            )
            .sort_values(["n", "POOLINV"])
            .reset_index(drop=True))

def endpoint_data_audit(data: pd.DataFrame) -> None:
    print(f"Shape: {data.shape} | Unique patients: {data['PATIENT'].nunique()}\n")

    print("--- Therapy / Gender balance ---")
    display(data[["THERAPY", "GENDER"]].value_counts().unstack(fill_value=0))

    print("\n--- Metric statistics ---")
    display(data[["BASVAL", "HAMDTL17", "CHANGE", "percentage_response"]].describe().T)


def distribution_diagnostics(data: pd.DataFrame, column: str) -> pd.DataFrame:
    """Distribution diagnostics for a variable."""
    values = data[column].dropna()

    stats_dict = {
        "variable": column,
        "n": len(values),
        "mean": values.mean(),
        "median": values.median(),
        "std": values.std(),
        "iqr": values.quantile(0.75) - values.quantile(0.25),
        "min": values.min(),
        "max": values.max(),
        "skewness": stats.skew(values),
        "kurtosis": stats.kurtosis(values),
    }

    if len(values) >= 8:
        k2, p = stats.normaltest(values)
        stats_dict["normaltest_p"] = p
    else:
        stats_dict["normaltest_p"] = np.nan

    return pd.DataFrame([stats_dict])


# ================= MODELING =================

def fit_adjustment(data: pd.DataFrame, target: str = "percentage_response"):
    cols = [target, "treatment", "BASVAL_centered", "gender_female", "POOLINV"]
    df = data[cols].dropna().astype({"treatment": float, "BASVAL_centered": float, "gender_female": float})
    f = f"{target} ~ treatment + BASVAL_centered + gender_female"

    try:
        m = smf.mixedlm(f, df, groups=df["POOLINV"]).fit(reml=False, method="lbfgs")
        return m, df, "MixedLM"
    except Exception as e:
        print(f"MixedLM failed: {e}. Using OLS fallback.")
        m = smf.ols(f"{f} + C(POOLINV)", df).fit(cov_type="HC3")
        return m, df, "OLS_FE_HC3"


def add_residuals(data: pd.DataFrame, model, model_data: pd.DataFrame,
                  target: str, col: str = "adjusted_residual") -> pd.DataFrame:
    return data.assign(**{col: data[target] - model.predict(model_data)})


def fit_gmm(values: np.ndarray, n_init: int = 50,
            components=range(1, 6), cov_types=("full", "tied", "diag", "spherical"),
            random_state: int = 42) -> pd.DataFrame:
    x = np.asarray(values, dtype=float).reshape(-1, 1)
    x = x[np.isfinite(x).ravel()]
    if not len(x): raise ValueError("No finite observations.")

    rng = np.random.default_rng(random_state)
    rows = []

    for k in components:
        for cov in cov_types:
            best_m, best_ll = None, -np.inf
            for seed in rng.integers(0, 2**31-1, n_init):
                try:
                    m = GaussianMixture(k, covariance_type=cov, random_state=int(seed),
                                      n_init=1, reg_covar=1e-6).fit(x)
                    if m.converged_ and (ll := m.score(x)*len(x)) > best_ll:
                        best_ll, best_m = ll, m
                except: continue
            if best_m:
                rows.append({"n_components": k, "covariance_type": cov,
                           "bic": best_m.bic(x), "model": best_m})

    return pd.DataFrame(rows).sort_values("bic").reset_index(drop=True)


def summarize_gmm(model: GaussianMixture) -> pd.DataFrame:
    if model.covariance_type == "full":
        var = model.covariances_[:, 0, 0]
    elif model.covariance_type == "tied":
        var = np.full(model.n_components, model.covariances_[0, 0])
    else:
        var = model.covariances_.ravel()

    return (pd.DataFrame({
                "component": np.arange(1, model.n_components + 1),
                "weight": model.weights_,
                "mean": model.means_.ravel(),
                "std": np.sqrt(var)
            })
            .sort_values("mean")
            .reset_index(drop=True)
            .assign(weight_pct=lambda x: x.weight*100, stable=lambda x: x.weight >= 0.05))


def bootstrap_gmm(values: np.ndarray, n_boot: int = 50, n_init: int = 30,
                  random_state: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(random_state)
    rows = []
    for i in range(n_boot):
        sample = rng.choice(values, len(values), replace=True)
        try:
            res = fit_gmm(sample, n_init=n_init, random_state=random_state + i)
            best = res.iloc[0]
            rows.append({"boot_id": i, "k": int(best["n_components"]),
                        "bic": float(best["bic"]), "cov": best["covariance_type"]})
        except Exception as e:
            rows.append({"boot_id": i, "k": pd.NA, "bic": np.nan, "cov": None, "error": str(e)})
    return pd.DataFrame(rows)


def fit_qr(data: pd.DataFrame, formula: str, target: str = "percentage_response",
           quantiles: list[float] | None = None, name: str = "model") -> pd.DataFrame:
    if quantiles is None: quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
    df = data.dropna(subset=[target, "treatment", "BASVAL_centered", "gender_female", "POOLINV"]).copy()
    rows = []

    for q in quantiles:
        try:
            m = smf.quantreg(formula, df).fit(q=q, max_iter=5000)
            null = smf.quantreg(f"{target} ~ 1", df).fit(q=q, max_iter=5000)
            loss = pinball_loss(df[target], m.predict(df), q)
            null_loss = pinball_loss(df[target], null.predict(df), q)
            ci = m.conf_int().loc["treatment"]


            rows.append({
                "model_name": name,
                "quantile": q,
                "treatment_coef": m.params.get("treatment", np.nan),
                "ci_low": ci[0],
                "ci_high": ci[1],
                "p_value": m.pvalues.get("treatment", np.nan),
                "pinball_loss_improvement": 1 - loss / null_loss if null_loss != 0 else np.nan,
                "n": len(df)
            })
        except Exception as e:
            rows.append({"model_name": name, "quantile": q, "error": str(e), "n": len(df)})


    return add_fdr(pd.DataFrame(rows))


# ================= PLOTS =================

def plot_gmm(values: np.ndarray, model: GaussianMixture, filename: Optional[str] = None):
    x = np.linspace(values.min(), values.max(), 1000).reshape(-1, 1)
    dens = np.exp(model.score_samples(x))
    plt.figure(figsize=(9, 5))
    sns.histplot(values, bins=25, stat="density", alpha=0.35, label="Observed")
    plt.plot(x[:, 0], dens, lw=2, label="FMM")
    for i in range(model.n_components):
        plt.plot(x[:, 0], model.predict_proba(x)[:, i] * dens, "--", label=f"Comp {i+1}")
    finalize_plot("GMM Density", "Adjusted residual", "Density", (9, 5), filename)



def plot_gmm_selection(gmm_res: pd.DataFrame, title: str, filename: str | None = None):
    best = gmm_res.groupby("n_components", as_index=False)["bic"].min()

    plt.figure(figsize=(7, 5))
    sns.lineplot(data=best, x="n_components", y="bic", marker="o")

    finalize_plot(title, "Number of components", "Best BIC", (7, 5), filename)


def plot_bootstrap(boot: pd.DataFrame, filename: Optional[str] = None):
    plt.figure(figsize=(7, 5))
    sns.countplot(data=boot, x="k")
    finalize_plot("Bootstrap Stability", "Selected K", "Count", (7, 5), filename)



def plot_qr(qr: pd.DataFrame, title: str = "Quantile Regression", filename: str | None = None):
    plt.figure(figsize=(8, 5))
    plt.plot(qr["quantile"], qr["treatment_coef"], marker="o", label="Treatment coef")
    if "ci_low" in qr.columns:
        plt.fill_between(qr["quantile"], qr["ci_low"], qr["ci_high"], alpha=0.2, label="95% CI")

    plt.axhline(0, ls="--", color="gray", lw=1)
    finalize_plot(title, "Quantile", "Treatment coefficient", (8, 5), filename)



def plot_qr_compare(qr: pd.DataFrame, y: str, title: str = "QR Comparison",
                    ylabel: str = "Value", filename: Optional[str] = None):
    plt.figure(figsize=(9, 5))
    for name, g in qr.dropna(subset=[y]).groupby("model"):
        plt.plot(g["q"], g[y], marker="o", label=name)
    if "coef" in y: plt.axhline(0, ls="--", color="gray")
    finalize_plot(title, "Quantile", ylabel, (9, 5), filename)


print("Functions loaded.")

# 6. Initial data audit

Goal of this step: understand the structure of the source data, check types, missing values, visits, treatment arms, and the basic logic of the `CHANGE` variable.

In [ ]:
df = prepare_data(df_raw)

print("Shape:", df.shape)
print("Unique patients:", df["PATIENT"].nunique())
print("Columns:", df.columns.tolist())

print("\nData types:")
display(df.dtypes)

print("\nTreatment counts by rows:")
display(df["THERAPY"].value_counts())

print("\nVisit counts by rows:")
display(df["VISIT"].value_counts().sort_index())

print("\nPOOLINV counts by rows:")
display(df["POOLINV"].value_counts().sort_index())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

display(df.head())

In [ ]:
# check the Change column
df["manual_change"] = df["HAMDTL17"] - df["BASVAL"]
max_abs_difference = (df["CHANGE"] - df["manual_change"]).abs().max()

print("Max absolute difference between CHANGE and HAMDTL17 - BASVAL:", max_abs_difference)
display(df[["PATIENT", "BASVAL", "HAMDTL17", "CHANGE", "manual_change"]].head(100))

## Data audit conclusion

 The dataset is in long format: one row is a patient visit, not a single patient. CHANGE = HAMDTL17 - BASVAL.
 Improvement in depression corresponds to a decrease in HAMD17, so for convenience absolute_response = -CHANGE is used, where a larger value means greater improvement.

# 7. Data processing and creation of outcome endpoints.

Here the longitudinal dataset is converted into an endpoint dataset: one row per patient.

## Primary endpoint

The last available patient visit (last_available) is used. This reduces data loss due to dropout.

## Primary HAMD score threshold

BASVAL >= 14 is used; additional thresholds are tested in the sensitivity analysis.

In [ ]:
df_endpoint = build_endpoint(
    df,
    endpoint="last_available",
    baseline_cutoff=PRIMARY_BASELINE_CUTOFF,
)

df_visit7 = build_endpoint(
    df,
    endpoint="visit7",
    baseline_cutoff=PRIMARY_BASELINE_CUTOFF,
)

print("Primary endpoint: last available visit")
print("Primary baseline cut-off:", PRIMARY_BASELINE_CUTOFF)
print("Primary endpoint shape:", df_endpoint.shape)
print("Visit 7 endpoint shape:", df_visit7.shape)

display(df_endpoint.head())

In [ ]:
endpoint_data_audit(df_endpoint)

# 8. EDA: data exploration


For BASVAL and HAMDTL17, bins of **one point** wide are used because HAMD-17 is a discrete clinical scale. A KDE curve is not used here to avoid creating artificial smoothness on integer scores.

In [ ]:
x = df_endpoint["BASVAL"].dropna()
bins = np.arange(x.min() - 0.5, x.max() + 1.5, 1)

# draw the histogram
sns.histplot(
    x,
    bins=bins,
    stat="count",
    edgecolor="black",
    alpha=0.75,
)

# create the cut-off vertical line
plt.axvline(
    PRIMARY_BASELINE_CUTOFF,
    color="darkblue",
    linestyle="--",
    linewidth=2,
    label=f"Cut-off = {PRIMARY_BASELINE_CUTOFF}",
)


plt.grid(axis="y", alpha=0.25)

finalize_plot(
    title="Distribution of baseline HAMD-17 scores",
    xlabel="Baseline HAMD-17 score",
    ylabel="Number of patients",
    figsize=(10, 6),
    filename="01_baseline_hamd17_distribution.png"
)

In [ ]:
x = df_endpoint["HAMDTL17"].dropna()
bins = np.arange(x.min() - 0.5, x.max() + 1.5, 1)

# draw the histogram itself
sns.histplot(
    x,
    bins=bins,
    stat="count",
    edgecolor="black",
    alpha=0.75,
)


plt.grid(axis="y", alpha=0.25)


finalize_plot(
    title="Distribution of endpoint HAMD-17 scores",
    xlabel="Endpoint HAMD-17 score",
    ylabel="Number of patients",
    figsize=(10, 6),
    filename="02_endpoint_hamd17_distribution.png"
)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_endpoint["absolute_response"], bins=25, kde=True)
plt.title("Absolute response distribution")
plt.xlabel("Absolute response = baseline HAMD17 - endpoint HAMD17")
plt.ylabel("Number of patients")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_endpoint["percentage_response"], bins="fd", kde=True)
plt.title("Percentage response distribution")
plt.xlabel("Percentage response")
plt.ylabel("Number of patients")

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=df_endpoint, x="THERAPY", y="percentage_response")
plt.title("Percentage response by treatment group")
plt.xlabel("Treatment group")
plt.ylabel("Percentage response")

plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(data=df_endpoint, x="THERAPY", y="percentage_response", inner="quartile")
plt.title("Percentage response distribution by treatment group")
plt.xlabel("Treatment group")
plt.ylabel("Percentage response")
plt.show()

In [ ]:
distribution_summary = pd.concat(
    [
        distribution_diagnostics(df_endpoint, "absolute_response"),
        distribution_diagnostics(df_endpoint, "percentage_response"),
    ],
    ignore_index=True,
)

display(distribution_summary)

distribution_summary.to_csv(
    RESULTS_DIR / "distribution_summary_primary_endpoint.csv",
    index=False,
)

## EDA conclusion

Treatment effect: in the DRUG group the median response is noticeably higher than in PLACEBO, but there is strong individual spread (from worsening to full remission). Distribution shape: the response metrics (absolute and percentage) show no critical anomalies (D'Agostino $p > 0.05$) but display wide tails. Conclusion: the high dispersion of response visually confirms that comparing only means is insufficient. The data contain heterogeneity, which fully justifies the subsequent use of FMM (to look for subgroups) and quantile regression.

# 9. `POOLINV` diagnostics

This section is needed to justify:

1. why we account for POOLINV;
2. why MixedLM may fail to converge;

In [ ]:
poolinv_summary = poolinv_diag(df_endpoint)

display(poolinv_summary)
display(poolinv_summary.describe(include="all").T)

poolinv_summary.to_csv(
    RESULTS_DIR / "poolinv_diagnostics.csv",
    index=False,
)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(poolinv_summary["n"], bins=10)
plt.title("Number of patients per POOLINV group")
plt.xlabel("Patients per POOLINV")
plt.ylabel("Number of POOLINV groups")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(poolinv_summary["drug_share"], bins=10)
plt.title("Treatment balance inside POOLINV groups")
plt.xlabel("Drug share within POOLINV")
plt.ylabel("Number of POOLINV groups")
plt.show()

## `POOLINV` conclusion

 The structure confirms that POOLINV does reflect the level of individual investigator sites. However, the small group sizes make mixed-effects models (random-intercept MixedLM) computationally unstable. To work around this, the pipeline has a fallback — switching to a specification with site fixed effects and HC3 robust errors: response ~ treatment + BASVAL_centered + gender_female + C(POOLINV).

The POOLINV variable does its job well for regression adjustment, but the sizes of these groups are too small to try to interpret them as fully independent trials.

# 10. Model 1: baseline and POOLINV adjustment

The goal of this model is to compute adjusted residuals. We isolate the part of individual response that does not depend on treatment assignment, baseline depression severity, patient gender, and the specific site effect. The resulting residuals are then fed into the FMM model.

In [ ]:
adjustment_result, adjustment_data, adjustment_model_type = fit_adjustment(
    df_endpoint,
    target="percentage_response",
)

# Preserve primary percentage-response adjustment results.
# This avoids accidental overwriting later in sensitivity sections.
primary_adjustment_result_percentage = adjustment_result
primary_adjustment_data_percentage = adjustment_data.copy()
primary_adjustment_model_type_percentage = adjustment_model_type

print("Adjustment model type:", adjustment_model_type)
print(adjustment_result.summary())

In [ ]:
df_endpoint = add_residuals(
    data=df_endpoint,
    model=adjustment_result,
    model_data=adjustment_data,
    target="percentage_response",
    col="adjusted_residual_percentage",
)

display(df_endpoint[["percentage_response", "adjusted_residual_percentage"]].head())

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_endpoint["adjusted_residual_percentage"].dropna(), bins=25, kde=True)
plt.title("Adjusted residual distribution: percentage response")
plt.xlabel("Adjusted residual")
plt.ylabel("Number of patients")
plt.show()

In [ ]:
sm.qqplot(df_endpoint["adjusted_residual_percentage"].dropna(), line="45")
plt.title("QQ-plot of adjusted residuals: percentage response")
plt.show()

In [ ]:
residual_diagnostics = distribution_diagnostics(
    df_endpoint,
    "adjusted_residual_percentage",
)

display(residual_diagnostics)

residual_diagnostics.to_csv(
    RESULTS_DIR / "adjusted_residual_diagnostics_percentage.csv",
    index=False,
)

## Adjusted model conclusion

The residuals look reasonably normal: they are centered around zero, show no strong skewness, and the normality test does not reject normality. This matters.

# 11. Model 2: Finite Mixture Models

Finite mixture models let us test whether the distribution of residuals can be represented as a combination of several normal components. In this work the Gaussian mixture model is applied to one-dimensional residuals, not to the full set of patient features.


The Gaussian mixture model is fit with the EM algorithm. To reduce the dependence of the result on local optima, each candidate model is started from several random initial states. For a given number of components and covariance type, the solution with the highest log-likelihood among all successful runs is kept.

Models with different numbers of components (K) are then compared by the Bayesian information criterion (BIC). A lower BIC means a better trade-off between goodness of fit and model complexity.

Additionally, the following are assessed:

    the size of the identified components;

    the stability of the choice of K via bootstrap;

    the sensitivity of the results to a change of the target metric, the choice of trial endpoint, and the baseline severity cut-off.

In [ ]:
fmm_values = df_endpoint["adjusted_residual_percentage"].dropna().values


fmm_values = df_endpoint["adjusted_residual_percentage"].dropna().values

gmm_results_percentage = fit_gmm(
    values=fmm_values,
    components=FMM_COMPONENTS,
    cov_types=FMM_COVARIANCE_TYPES_MAIN,
    random_state=RANDOM_STATE,
    n_init=50
)

display(gmm_results_percentage.head())

best_gmm_percentage = gmm_results_percentage.loc[0, "model"]

# Preserve primary percentage-response FMM results.
primary_gmm_results_percentage = gmm_results_percentage.copy()
primary_best_gmm_percentage = best_gmm_percentage

print("Best FMM by BIC:")
print("K:", gmm_results_percentage.loc[0, "n_components"])
print("Covariance:", gmm_results_percentage.loc[0, "covariance_type"])
print("BIC:", gmm_results_percentage.loc[0, "bic"])



In [ ]:
plot_gmm_selection(
    gmm_results_percentage,
    title="FMM model selection: percentage response residuals",
    filename="11_fmm_model_selection_percentage.png",
)

In [ ]:
gmm_summary_percentage = summarize_gmm(best_gmm_percentage)

# save the primary response results
primary_gmm_summary_percentage = gmm_summary_percentage.copy()

display(gmm_summary_percentage)

In [ ]:
plot_gmm(
    values=fmm_values,
    model=best_gmm_percentage,
    filename="12_fmm_density_percentage.png",
)

## 11.3. Checking tails and outliers

If the FMM selects an additional small component, we need to check whether it is created by a few extreme observations.


In [ ]:
outlier_table = df_endpoint.sort_values("adjusted_residual_percentage").head(10)[
    [
        "PATIENT",
        "THERAPY",
        "GENDER",
        "POOLINV",
        "BASVAL",
        "HAMDTL17",
        "CHANGE",
        "absolute_response",
        "percentage_response",
        "adjusted_residual_percentage",
        "VISIT",
        "RELDAYS",
    ]
]

display(outlier_table)

outlier_table.to_csv(
    RESULTS_DIR / "lowest_adjusted_residuals_inspection.csv",
    index=False,
)

# 12. FMM validation via bootstrap

Bootstrap is used here to check the stability of the choice of the number of components. If different K are selected on different bootstrap samples, the latent subgroups cannot be considered stable.

In [ ]:
fmm_bootstrap_percentage = bootstrap_gmm(
    fmm_values,
    random_state=RANDOM_STATE,
)

primary_fmm_bootstrap_percentage = fmm_bootstrap_percentage.copy()

display(fmm_bootstrap_percentage["k"].value_counts(dropna=False).sort_index())

# save the result
fmm_bootstrap_percentage.to_csv(
    RESULTS_DIR / "fmm_bootstrap_stability_percentage.csv",
    index=False,
)

In [ ]:
plot_bootstrap(
    fmm_bootstrap_percentage,

    filename="13_fmm_bootstrap_selected_k_percentage.png",
)

## FMM validation conclusion
In the primary analysis, the best BIC model had 1 component; on the bootstrap samples this value was also selected most often — 136/150. Additional components are almost never selected, so K=1 can be considered stable.
Adding extra components increased BIC, i.e. it did not give a sufficient improvement in goodness of fit once the complexity penalty was accounted for. The bootstrap analysis confirmed the stability of the result.

FMM did not reveal a stable multi-component structure.

# 13. Model 3: Quantile regression


Quantile regression estimates the response not at the mean but at different quantiles of the response distribution.



Since we have an individual-patient-data study, including covariates is an advantage.

In [ ]:
qr_formula_percentage = "percentage_response ~ treatment + BASVAL_centered + gender_female"

qr_results_percentage = fit_qr(
    data=df_endpoint,
    formula=qr_formula_percentage,
    target="percentage_response",
    quantiles=QUANTILES,
    name="baseline_gender",
)

display(qr_results_percentage)

In [ ]:
plot_qr(
    qr_results_percentage,
    filename="14_quantile_regression_percentage_primary.png",
)

## 13.2. Goodness of fit for quantile regression

Quantile regression has no ordinary OLS R². Therefore the following are used:

**pinball loss** — the main loss function;
**pseudo R²** — a relative quality measure;
**stability of treatment coefficient** — stability of the treatment-effect profile when covariates are added.

Three specifications are compared:

1. `treatment_only` — treatment only,
2. `baseline_gender` — gender,
3. `baseline_gender_poolinv` — treatment + gender

In [ ]:
qr_model_specs_percentage = {
    "treatment_only": "percentage_response ~ treatment",
    "baseline_gender": "percentage_response ~ treatment + BASVAL_centered + gender_female",
    "baseline_gender_poolinv": (
        "percentage_response ~ treatment + BASVAL_centered + gender_female + C(POOLINV)"
    ),
}

qr_comparison_percentage = []

for model_name, formula in qr_model_specs_percentage.items():
    qr_comparison_percentage.append(
        fit_qr(
            data=df_endpoint,
            formula=formula,
            target="percentage_response",
            quantiles=QUANTILES,
            name=model_name,
        )
    )

qr_comparison_percentage = pd.concat(qr_comparison_percentage, ignore_index=True)

display(qr_comparison_percentage)

## Conclusion

Quantile regression showed that the treatment effect differs across patients with different strengths of improvement. For patients with little improvement, the difference between drug and placebo is small. For patients with moderate and fairly strong improvement, this difference is larger. At the most extreme parts of the distribution the conclusions are less reliable because there are fewer observations and the uncertainty intervals are wider. An additional check showed that accounting for POOLINV improves the description of the data: the model gives a lower error and explains differences in response better. This means that the investigator or site group is indeed related to how the patient response is distributed.

The overall conclusion is: patients respond to treatment differently, but this does not necessarily mean they split into several clear hidden groups. Together with the FMM results, this gives the following picture: no stable separate response groups were found, but differences in the strength of improvement between patients do exist.

# 14. Sensitivity analysis

In this project, the sensitivity analysis checks the robustness of the results to several choices:

1. change of target: percentage_response → absolute_response;
2. change of endpoint: last_available → VISIT == 7;
3. change of baseline threshold: no cut-off, 14, 18, 20, 24;
4. comparison of FMM and quantile regression across scenarios;
5. spline sensitivity for baseline severity;
6. an ANCOVA model as an additional check of the result.

## 14.1. Sensitivity target: absolute response

In [ ]:

absolute_adjustment_result, absolute_adjustment_data, absolute_model_type = fit_adjustment(
    df_endpoint,
    target="absolute_response"
)

print("Adjustment model type:", absolute_model_type)
print(absolute_adjustment_result.summary())

In [ ]:
df_endpoint = add_residuals(
    df_endpoint,
    absolute_adjustment_result,
    absolute_adjustment_data,
    target="absolute_response",
    col="adjusted_residual_absolute"
)

In [ ]:
absolute_fmm_values = df_endpoint["adjusted_residual_absolute"].dropna().values

gmm_results_absolute = fit_gmm(
    absolute_fmm_values,
    n_init=FMM_RANDOM_STARTS_MAIN
)

available_cols = gmm_results_absolute.columns.tolist()
desired_cols = ["n_components", "covariance_type", "bic", "aic", "converged_runs", "best_seed"]
display_cols = [col for col in desired_cols if col in available_cols]

print(f"Available columns: {available_cols}")
display(gmm_results_absolute[display_cols].head(10))


best_gmm_absolute = gmm_results_absolute.loc[0, "model"]
gmm_summary_absolute = summarize_gmm(best_gmm_absolute)

primary_gmm_results_absolute = gmm_results_absolute.copy()
primary_best_gmm_absolute = best_gmm_absolute
primary_gmm_summary_absolute = gmm_summary_absolute.copy()

display(gmm_summary_absolute)


gmm_results_absolute[display_cols].to_csv(
    RESULTS_DIR / "fmm_model_selection_absolute.csv",
    index=False,
)

gmm_summary_absolute.to_csv(
    RESULTS_DIR / "fmm_component_summary_absolute.csv",
    index=False,
)

In [ ]:
# GMM model-selection plot
plot_gmm_selection(
    gmm_results_absolute,
    title="FMM model selection: absolute response residuals",
    filename="18_fmm_model_selection_absolute.png",
)

# density plot
plot_gmm(
    values=absolute_fmm_values,
    model=best_gmm_absolute,
    filename="19_fmm_density_absolute.png",
)

In [ ]:
fmm_bootstrap_absolute = bootstrap_gmm(
    absolute_fmm_values,
    n_boot=FMM_BOOTSTRAP_ITERATIONS,
    n_init=FMM_RANDOM_STARTS_MAIN,
    random_state=RANDOM_STATE
)

primary_fmm_bootstrap_absolute = fmm_bootstrap_absolute.copy()


print("Distribution of the selected number of components (k):")
display(fmm_bootstrap_absolute["k"].value_counts(dropna=False).sort_index())


fmm_bootstrap_absolute.to_csv(
    RESULTS_DIR / "fmm_bootstrap_stability_absolute.csv",
    index=False,
)

In [ ]:
plot_bootstrap(
    fmm_bootstrap_absolute,
    filename="20_fmm_bootstrap_selected_k_absolute.png",
)

In [ ]:
qr_formula_absolute = "absolute_response ~ treatment + BASVAL_centered + gender_female"


qr_results_absolute = fit_qr(
    data=df_endpoint,
    formula=qr_formula_absolute,
    target="absolute_response",
    quantiles=QUANTILES,
    name="baseline_gender"
)


primary_qr_results_absolute = qr_results_absolute.copy()

display(qr_results_absolute)

qr_results_absolute.to_csv(
    RESULTS_DIR / "quantile_regression_absolute_primary.csv",
    index=False,
)


plot_qr(
    qr_results_absolute,
    title="Quantile regression: treatment effect across absolute response",
    filename="21_quantile_regression_absolute_primary.png"
)

In [ ]:
qr_comparison_absolute = []


for model_name, formula in qr_model_specs_absolute.items():
    qr_res = fit_qr(
        data=df_endpoint,
        formula=formula,
        target="absolute_response",
        quantiles=QUANTILES,
        name=model_name
    )
    qr_comparison_absolute.append(qr_res)


df_qr_comparison_absolute = pd.concat(qr_comparison_absolute, ignore_index=True)


df_qr_comparison_absolute.rename(
    columns={"model_name": "model", "quantile": "q"},
    inplace=True
)

display(df_qr_comparison_absolute)

df_qr_comparison_absolute.to_csv(
    RESULTS_DIR / "quantile_regression_absolute_comparison.csv",
    index=False,
)


plot_qr_compare(
    df_qr_comparison_absolute,
    y="treatment_coef",
    title="QR comparison: absolute response",
    ylabel="Treatment Coefficient",
    filename="22_quantile_regression_absolute_comparison.png"
)

In [ ]:
# treatment-coefficient comparison plot
plot_qr_compare(
    qr=df_qr_comparison_absolute,
    y="treatment_coef",
    title="QR model comparison: absolute response treatment coefficient",
    ylabel="Treatment coefficient",
    filename="22_qr_comparison_treatment_coef_absolute.png",
)

# quality-metric comparison plot (pinball loss improvement)
plot_qr_compare(
    qr=df_qr_comparison_absolute,
    y="pinball_loss_improvement",
    title="QR goodness-of-fit: absolute response pinball loss improvement",
    ylabel="Pinball loss improvement",
    filename="23_qr_comparison_pinball_loss_absolute.png",
)

## Summary of the absolute-response sensitivity analysis.

The absolute-response sensitivity analysis does not overturn the main conclusion of the work. In the primary percentage-response analysis there was no stable separation of patients into hidden groups. In the absolute response there is a hint of two components, but it is unstable. So the honest summary is: the data show unequal strength of response between patients, but do not give reliable evidence of stable hidden response groups.

## 14.2. Sensitivity to endpoint and baseline threshold

In [ ]:
sensitivity_percentage_rows = []

for endpoint_name in ["last_available", "visit7"]:
    for cutoff in SENSITIVITY_CUTOFFS:
        try:
            # Filter the data for a specific threshold and endpoint
            df_ep = build_endpoint(df, endpoint=endpoint_name, baseline_cutoff=cutoff)

            # fit the adjustment model
            m, df_model, m_name = fit_adjustment(df_ep, target="percentage_response")
            trt_coef = m.params.get("treatment", None)
            trt_pval = m.pvalues.get("treatment", None)

            # compute the residuals
            df_ep = add_residuals(df_ep, m, df_model, target="percentage_response", col="res")
            res_vals = df_ep["res"].dropna().values
            n_obs = len(res_vals)

            # fit the GMM
            gmm_res = fit_gmm(res_vals, n_init=FMM_RANDOM_STARTS_MAIN)
            best_k = gmm_res.loc[0, "n_components"]
            best_bic = gmm_res.loc[0, "bic"]

        except Exception as e:
            print(f"Skipped {endpoint_name} (cutoff {cutoff}): {e}")
            trt_coef = trt_pval = best_k = best_bic = None
            n_obs = len(df_ep) if 'df_ep' in locals() else 0


        sensitivity_percentage_rows.append({
            "endpoint": endpoint_name,
            "cutoff": cutoff if cutoff is not None else "None",
            "target": "percentage_response",
            "n_obs": n_obs,
            "treatment_coef": trt_coef,
            "treatment_pval": trt_pval,
            "best_k": best_k,
            "best_bic": best_bic
        })


sensitivity_percentage = pd.DataFrame(sensitivity_percentage_rows)

display(sensitivity_percentage)

sensitivity_percentage.to_csv(
    RESULTS_DIR / "sensitivity_summary_percentage.csv",
    index=False,
)

In [ ]:
sensitivity_absolute_rows = []

for endpoint_name in ["last_available", "visit7"]:
    for cutoff in SENSITIVITY_CUTOFFS:
        try:

            df_ep = build_endpoint(df, endpoint=endpoint_name, baseline_cutoff=cutoff)


            m, df_model, m_name = fit_adjustment(df_ep, target="absolute_response")
            trt_coef = m.params.get("treatment", None)
            trt_pval = m.pvalues.get("treatment", None)


            df_ep = add_residuals(df_ep, m, df_model, target="absolute_response", col="res")
            res_vals = df_ep["res"].dropna().values
            n_obs = len(res_vals)


            gmm_res = fit_gmm(res_vals, n_init=FMM_RANDOM_STARTS_MAIN)
            best_k = gmm_res.loc[0, "n_components"]
            best_bic = gmm_res.loc[0, "bic"]

        except Exception as e:
            print(f"Skipped {endpoint_name} (cutoff {cutoff}): {e}")
            trt_coef = trt_pval = best_k = best_bic = None
            n_obs = len(df_ep) if 'df_ep' in locals() else 0


        sensitivity_absolute_rows.append({
            "endpoint": endpoint_name,
            "cutoff": cutoff if cutoff is not None else "None",
            "target": "absolute_response",
            "n_obs": n_obs,
            "treatment_coef": trt_coef,
            "treatment_pval": trt_pval,
            "best_k": best_k,
            "best_bic": best_bic
        })


sensitivity_absolute = pd.DataFrame(sensitivity_absolute_rows)

display(sensitivity_absolute)

sensitivity_absolute.to_csv(
    RESULTS_DIR / "sensitivity_summary_absolute.csv",
    index=False,
)

In [ ]:
print("Percentage response sensitivity pivot:")
display(
    sensitivity_percentage.pivot_table(
        index=["endpoint", "cutoff"],
        values=["n_obs", "treatment_coef", "treatment_pval", "best_k", "best_bic"],
        aggfunc="first",
    )
)

print("Absolute response sensitivity pivot:")
display(
    sensitivity_absolute.pivot_table(
        index=["endpoint", "cutoff"],
        values=["n_obs", "treatment_coef", "treatment_pval", "best_k", "best_bic"],
        aggfunc="first",
    )
)

## Conclusion of the baseline-threshold sensitivity

The sensitivity analysis showed that the treatment effect stays positive in most variants: the DRUG group usually improved more than PLACEBO. Without a baseline-severity threshold the result becomes weaker, so BASVAL >= 14 looks like a reasonable primary choice: it removes overly mild cases but keeps a sufficient sample size. For the percentage response the mixture model almost always selects a single component, i.e. there is no visible stable separation of patients into hidden groups. For the absolute response two components sometimes appear, but this result depends on the chosen endpoint and threshold, so it is better treated as a weak and unstable signal rather than evidence of separate patient groups.

## 14.3. Sensitivity: spline baseline adjustment and ANCOVA-style endpoint model

In Stone et al., multivariable adaptive regression splines were used to capture non-linear effects of continuous covariates. In this work the primary analysis is intentionally simpler because of the small sample size,
but a sensitivity analysis with a spline transform of BASVAL_centered is added.

This section answers two questions:

1. Does the treatment coefficient change much if baseline severity enters non-linearly?
2. Does the conclusion hold if we model not change/response but the endpoint HAMD-17 score
   in an ANCOVA-style specification?


In [ ]:
linear_response_formula = (
    "percentage_response ~ treatment + BASVAL_centered + "
    "gender_female + C(POOLINV)"
)

spline_response_formula = (
    "percentage_response ~ treatment + "
    "bs(BASVAL_centered, df=3, degree=3) + "
    "gender_female + C(POOLINV)"
)

linear_response_model = smf.ols(
    formula=linear_response_formula,
    data=df_endpoint,
).fit(cov_type="HC3")

spline_response_model = smf.ols(
    formula=spline_response_formula,
    data=df_endpoint,
).fit(cov_type="HC3")


def extract_treatment_row(model_result, model_name: str) -> dict:
    """Extract treatment coefficient summary from a statsmodels result."""
    conf_int = model_result.conf_int()
    treatment_coef = model_result.params.get("treatment", np.nan)

    return {
        "model_name": model_name,
        "treatment_coef": treatment_coef,
        "ci_low": conf_int.loc["treatment", 0],
        "ci_high": conf_int.loc["treatment", 1],
        "p_value": model_result.pvalues.get("treatment", np.nan),
        "aic": model_result.aic,
        "bic": model_result.bic,
        "n": int(model_result.nobs),
    }


spline_adjustment_comparison = pd.DataFrame(
    [
        extract_treatment_row(linear_response_model, "linear_baseline_poolinv"),
        extract_treatment_row(spline_response_model, "spline_baseline_poolinv"),
    ]
)

display(spline_adjustment_comparison)

spline_adjustment_comparison.to_csv(
    RESULTS_DIR / "spline_adjustment_comparison_percentage.csv",
    index=False,
)

In [ ]:
qr_formula_spline_percentage = (
    "percentage_response ~ treatment + "
    "bs(BASVAL_centered, df=3, degree=3) + "
    "gender_female"
)

qr_results_spline_percentage = fit_qr(
    data=df_endpoint,
    formula=qr_formula_spline_percentage,
    target="percentage_response",
    quantiles=QUANTILES,
    name="baseline_spline_gender",
)


qr_linear_vs_spline_percentage = pd.concat(
    [

        qr_results_percentage.assign(model_name="baseline_linear_gender"),
        qr_results_spline_percentage,
    ],
    ignore_index=True,
)

display(qr_results_spline_percentage)


qr_results_spline_percentage.to_csv(
    RESULTS_DIR / "quantile_regression_percentage_spline_sensitivity.csv",
    index=False,
)

qr_linear_vs_spline_percentage.to_csv(
    RESULTS_DIR / "quantile_regression_linear_vs_spline_percentage.csv",
    index=False,
)

In [ ]:

qr_linear_vs_spline_percentage.rename(
    columns={"model_name": "model", "quantile": "q"},
    inplace=True
)


plot_qr_compare(
    qr=qr_linear_vs_spline_percentage,
    y="treatment_coef",
    title="QR sensitivity: linear vs spline baseline adjustment",
    ylabel="Treatment coefficient",
    filename="24_qr_linear_vs_spline_treatment_coef_percentage.png",
)


plot_qr_compare(
    qr=qr_linear_vs_spline_percentage,
    y="pinball_loss_improvement", # <-- use the current metric
    title="QR sensitivity: linear vs spline pinball loss improvement",
    ylabel="Pinball loss improvement",
    filename="25_qr_linear_vs_spline_pinball_loss_percentage.png",
)

In [ ]:
endpoint_linear_formula = (
    "HAMDTL17 ~ treatment + BASVAL_centered + "
    "gender_female + C(POOLINV)"
)

endpoint_spline_formula = (
    "HAMDTL17 ~ treatment + "
    "bs(BASVAL_centered, df=3, degree=3) + "
    "gender_female + C(POOLINV)"
)

endpoint_linear_model = smf.ols(
    formula=endpoint_linear_formula,
    data=df_endpoint,
).fit(cov_type="HC3")

endpoint_spline_model = smf.ols(
    formula=endpoint_spline_formula,
    data=df_endpoint,
).fit(cov_type="HC3")

ancova_endpoint_comparison = pd.DataFrame(
    [
        extract_treatment_row(endpoint_linear_model, "endpoint_linear_baseline_poolinv"),
        extract_treatment_row(endpoint_spline_model, "endpoint_spline_baseline_poolinv"),
    ]
)

ancova_endpoint_comparison[
    "implied_hamd17_improvement_for_drug"
] = -ancova_endpoint_comparison["treatment_coef"]

display(ancova_endpoint_comparison)

ancova_endpoint_comparison.to_csv(
    RESULTS_DIR / "ancova_endpoint_sensitivity.csv",
    index=False,
)

### Interpretation of the spline / ANCOVA sensitivity

The ANCOVA sensitivity confirms the main result: after adjusting for baseline severity, gender, and POOLINV, the drug group has a lower endpoint HAMD-17, by about 3.2 points compared with placebo. The spline check gives almost the same effect, so the conclusion does not depend on whether baseline severity is modeled linearly or a non-linear relationship is allowed.

# 15. Final results


In [ ]:
# 1. Re-create the function that formats the bootstrap distribution
def format_bootstrap(boot_df):
    if boot_df is None or boot_df.empty:
        return "N/A"
    # Count the proportions for each selected k
    counts = boot_df["k"].value_counts(normalize=True).sort_index()
    # Build a string like "k=1: 5%, k=2: 95%"
    return ", ".join([f"k={int(k)}: {pct:.0%}" for k, pct in counts.items()])

# 2. Get the median QR effects
# Check whether a q or quantile column exists
q_col_perc = "q" if "q" in qr_results_percentage.columns else "quantile"
q_col_abs = "q" if "q" in qr_results_absolute.columns else "quantile"
q_col_spline = "q" if "q" in qr_results_spline_percentage.columns else "quantile"

median_qr_percentage = qr_results_percentage.loc[
    np.isclose(qr_results_percentage[q_col_perc], 0.5), "treatment_coef"
]
median_qr_absolute = qr_results_absolute.loc[
    np.isclose(qr_results_absolute[q_col_abs], 0.5), "treatment_coef"
]
median_qr_spline_percentage = qr_results_spline_percentage.loc[
    np.isclose(qr_results_spline_percentage[q_col_spline], 0.5), "treatment_coef"
]

# 3. Deltas from ANCOVA/Spline (wrapped in try-except in case the tables were not recomputed)
try:
    linear_spline_delta = (
        spline_adjustment_comparison.loc[
            spline_adjustment_comparison["model_name"] == "spline_baseline_poolinv", "treatment_coef"
        ].iloc[0]
        - spline_adjustment_comparison.loc[
            spline_adjustment_comparison["model_name"] == "linear_baseline_poolinv", "treatment_coef"
        ].iloc[0]
    )
    endpoint_linear_benefit = ancova_endpoint_comparison.loc[
        ancova_endpoint_comparison["model_name"] == "endpoint_linear_baseline_poolinv", "implied_hamd17_improvement_for_drug"
    ].iloc[0]
    endpoint_spline_benefit = ancova_endpoint_comparison.loc[
        ancova_endpoint_comparison["model_name"] == "endpoint_spline_baseline_poolinv", "implied_hamd17_improvement_for_drug"
    ].iloc[0]
except NameError:
    linear_spline_delta = endpoint_linear_benefit = endpoint_spline_benefit = np.nan

# 4. Build the summary table
main_summary = pd.DataFrame([
    {
        "raw_rows": len(df),
        "unique_patients": df["PATIENT"].nunique(),
        "primary_endpoint": "last_available",
        "primary_baseline_cutoff": PRIMARY_BASELINE_CUTOFF,
        "primary_n": len(df_endpoint),
        "primary_n_drug": int((df_endpoint["THERAPY"] == "DRUG").sum()),
        "primary_n_placebo": int((df_endpoint["THERAPY"] == "PLACEBO").sum()),

        # Adjustment data (take m_name_perc from the updated fit_adjustment)
        "adjustment_model_type": m_name_perc if 'm_name_perc' in locals() else "LMM/OLS",
        "adjustment_treatment_coef_percentage": (
            m_perc.params.get("treatment", np.nan) if 'm_perc' in locals() else np.nan
        ),

        # GMM percentage data
        "best_fmm_k_percentage": primary_gmm_results_percentage.loc[0, "n_components"],
        "best_fmm_bic_percentage": primary_gmm_results_percentage.loc[0, "bic"],
        "smallest_fmm_component_weight_pct_percentage": primary_gmm_summary_percentage["weight_pct"].min(),
        "fmm_bootstrap_percentage": format_bootstrap(primary_fmm_bootstrap_percentage),
        "qr_median_treatment_coef_percentage": median_qr_percentage.iloc[0] if len(median_qr_percentage) else np.nan,

        # GMM absolute data
        "best_fmm_k_absolute": gmm_results_absolute.loc[0, "n_components"],
        "best_fmm_bic_absolute": gmm_results_absolute.loc[0, "bic"],
        "smallest_fmm_component_weight_pct_absolute": gmm_summary_absolute["weight_pct"].min(),
        "fmm_bootstrap_absolute": format_bootstrap(fmm_bootstrap_absolute),
        "qr_median_treatment_coef_absolute": median_qr_absolute.iloc[0] if len(median_qr_absolute) else np.nan,

        # Splines and other
        "qr_median_treatment_coef_spline_percentage": median_qr_spline_percentage.iloc[0] if len(median_qr_spline_percentage) else np.nan,
        "linear_vs_spline_adjustment_treatment_delta": linear_spline_delta,
        "ancova_endpoint_linear_hamd17_benefit": endpoint_linear_benefit,
        "ancova_endpoint_spline_hamd17_benefit": endpoint_spline_benefit,
        "bootstrap_iterations": FMM_BOOTSTRAP_ITERATIONS,
    }
])

# Output and save
display(main_summary.T) # transpose for readability (optional)

main_summary.to_csv(
    RESULTS_DIR / "main_summary.csv",
    index=False,
)

In [ ]:
print("Key results summary")
print("=" * 80)

# 1. Local function to format the bootstrap in place of the missing one
def format_bootstrap(boot_df):
    if boot_df is None or boot_df.empty:
        return "N/A"
    counts = boot_df["k"].value_counts(normalize=True).sort_index()
    return ", ".join([f"k={int(k)}: {pct:.0%}" for k, pct in counts.items()])

n_rows = len(df)
n_patients = df["PATIENT"].nunique()
n_primary = len(df_endpoint)

n_drug = (df_endpoint["THERAPY"] == "DRUG").sum()
n_placebo = (df_endpoint["THERAPY"] == "PLACEBO").sum()

print(f"Source data: {n_rows} rows, {n_patients} unique patients.")
print(
    "Primary analysis: last available visit, "
    f"BASVAL >= {PRIMARY_BASELINE_CUTOFF}."
)
print(f"Sample size: N = {n_primary} ({n_drug} DRUG, {n_placebo} PLACEBO).")

# Safely determine the adjustment model type
adj_model_type = primary_adjustment_model_type_percentage if 'primary_adjustment_model_type_percentage' in locals() else (m_name_perc if 'm_name_perc' in locals() else "MixedLM")
print(f"Adjustment model: {adj_model_type}.")

print("\nMixture model for percentage response")
best_k_percentage = int(primary_gmm_results_percentage.loc[0, "n_components"])
best_bic_percentage = primary_gmm_results_percentage.loc[0, "bic"]

print(f"Best number of components by BIC: {best_k_percentage}")
print(f"Best model BIC: {best_bic_percentage:.3f}")
print(f"Bootstrap stability: {format_bootstrap(primary_fmm_bootstrap_percentage)}")

print("\nMixture model for absolute response")
best_k_absolute = int(gmm_results_absolute.loc[0, "n_components"])
best_bic_absolute = gmm_results_absolute.loc[0, "bic"]

print(f"Best number of components by BIC: {best_k_absolute}")
print(f"Best model BIC: {best_bic_absolute:.3f}")
print(f"Bootstrap stability: {format_bootstrap(fmm_bootstrap_absolute)}")

if 'median_qr_percentage' in locals() and len(median_qr_percentage) > 0:
    median_qr_pct = median_qr_percentage.iloc[0]
    print(
        "\nQuantile regression, percentage response: "
        f"treatment coefficient at the median = {median_qr_pct:.4f}"
    )

if 'median_qr_absolute' in locals() and len(median_qr_absolute) > 0:
    median_qr_abs = median_qr_absolute.iloc[0]
    print(
        "Quantile regression, absolute response: "
        f"treatment coefficient at the median = {median_qr_abs:.4f}"
    )

if 'median_qr_spline_percentage' in locals() and len(median_qr_spline_percentage) > 0:
    median_qr_spline = median_qr_spline_percentage.iloc[0]
    print(
        "Quantile regression with spline adjustment: "
        f"treatment coefficient at the median = {median_qr_spline:.4f}"
    )

# Safely print the ANCOVA model results
print("\nANCOVA models for the endpoint HAMD-17")
if 'endpoint_linear_benefit' in locals() and not pd.isna(endpoint_linear_benefit):
    print(f"Linear baseline adjustment: {endpoint_linear_benefit:.3f} HAMD-17 points")
else:
    print("Linear baseline adjustment: N/A")

if 'endpoint_spline_benefit' in locals() and not pd.isna(endpoint_spline_benefit):
    print(f"Spline baseline adjustment: {endpoint_spline_benefit:.3f} HAMD-17 points")
else:
    print("Spline baseline adjustment: N/A")

print("\nFinal interpretation")
print(
    "For the primary response metric the mixture model most often selects "
    "a single component. The absolute-response analysis gives a similar result."
)
print(
    "Quantile regression shows that the difference between DRUG and PLACEBO "
    "changes across different parts of the response distribution."
)
print(
    "Taken together, the results do not support a stable separation "
    "of patients into several hidden response groups, but they do show differences "
    "in the strength of improvement between patients."
)

After reshaping the longitudinal data into a one-patient-per-row format and applying the baseline severity threshold BASVAL >= 14, the primary analysis included 136 patients: 70 in the DRUG group and 66 in the PLACEBO group.

For the pre-adjustment step, a linear mixed model with a random intercept by POOLINV was used. The model accounted for the treatment arm, baseline depression severity, and patient gender. The treatment coefficient for the percentage response was 0.137, i.e. after accounting for these factors the DRUG group had on average about a 13.7 percentage-point larger improvement than the PLACEBO group.

A Gaussian mixture model was then applied to the residuals of this model, i.e. to the part of individual response not explained by treatment, baseline severity, gender, and the POOLINV structure. For the primary metric percentage_response, the best BIC model had a single component: K = 1. This means the model found no need to split patients into several stable hidden response groups.

The bootstrap stability check confirmed this result: the one-component model was selected in 92% of the replicates. Models with two, three, and four components were selected much less often.

Quantile regression showed the other side of the result: the difference between DRUG and PLACEBO changes across different parts of the response distribution. At the median, the treatment coefficient for the percentage response was about 0.215, i.e. around 21.5 percentage points. This indicates that the strength of improvement is not the same across patients, even if stable hidden groups are not identified.

The sensitivity analysis on absolute_response broadly confirmed the main conclusion. When the percentage response was replaced by the change in HAMD-17 points, the best BIC model was again a single component. In the bootstrap, the single component was selected in 82% of the replicates. This shows that the conclusion of no stable separation into hidden groups is not an artifact of a single way of computing the response.

Final conclusion: in this open dataset, after accounting for the main factors, the Gaussian mixture model did not reveal a stable separation of patients into several hidden response groups. At the same time, quantile regression shows that the strength of response does differ between patients. The results therefore favor a cautious interpretation: individual heterogeneity of response exists, but in these data it is better described as a varying degree of improvement rather than as several clear patient types.

It is important to keep the limitations in mind: the work is based on a single open dataset with a small number of patients, a limited set of covariates, and an aggregated DRUG arm. The results should therefore be treated as exploratory EDA, not as clinical proof of the presence or absence of separate antidepressant response phenotypes.